# 06 — Performance benchmark

Benchmarks `AafragGammaSpectralModel.__call__`/`evaluate()` cost against (1) the number of
points in the *output* (gammapy-requested) energy grid, and (2) the number of
`primary_composition` x `target_composition` species pairs, separating **cold** calls
(first evaluation for a given `(primary, target, channel, energy_grid)` key -- includes the
expensive `aafragpy.get_cross_section` call) from **warm** calls (cache hit, ADR-003 --
only the cheap `get_spectrum` convolution runs). This repeats Step 0's benchmark
(ADR-013, which timed `aafragpy`'s two functions directly) at the `models.py` level, with
real composition dicts, to confirm ADR-002/ADR-003's "NumPy is fast enough, no LUT/JAX
needed for v1" assumption still holds through the full stack -- and to give a concrete,
numeric trigger point for revisiting it.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

from aafrag_gammapy import kernel, models

%matplotlib inline

## Benchmark 1: cost vs. output energy-grid size

The *internal* primary-energy integration grid (`N_PRIMARY_GRID_POINTS = 100`, ADR-018) is
fixed regardless of what energy grid gammapy actually requests -- so this tests whether the
final log-log interpolation onto an arbitrary-size output grid (`_interp_log_log`) is a
cost driver at realistic gammapy energy-axis sizes (typically a few tens of bins).

In [ ]:
pl = models.PowerLawParticleDistribution(amplitude="1e40 TeV-1", index=2.2)

grid_sizes = [10, 30, 100, 300, 1000]
n_reps = 20
results_grid = []

for n in grid_sizes:
    kernel._cross_section_cache.clear()
    model = models.AafragGammaSpectralModel(pl)
    energy = np.geomspace(0.1, 100, n) * u.TeV

    t0 = time.perf_counter()
    model(energy)
    t_cold = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(n_reps):
        model(energy)
    t_warm = (time.perf_counter() - t0) / n_reps

    results_grid.append((n, t_cold, t_warm))
    print(f"n_grid={n:5d}  cold={t_cold*1e3:8.1f} ms  warm={t_warm*1e3:7.3f} ms")

## Benchmark 2: cost vs. number of `(primary, target)` species pairs

Only `(p, H)`, `(p, He)`, `(He, p)`, `(He, He)` are jointly tabulated by AAfrag (ADR-012),
so the largest composition benchmarked here is the full 2x2 = 4-pair combination.

In [ ]:
pl_p = models.PowerLawParticleDistribution(amplitude="1e40 TeV-1", index=2.2)
pl_he = models.PowerLawParticleDistribution(amplitude="1e39 TeV-1", index=2.3)

compositions = [
    ("1 pair (p-H)", pl_p, {"H": 1.0}),
    ("2 pairs (p-H, p-He)", pl_p, {"H": 1.0, "He": 0.1}),
    ("4 pairs (p+He x H+He)", {"p": pl_p, "He": pl_he}, {"H": 1.0, "He": 0.1}),
]

energy = np.geomspace(0.1, 100, 30) * u.TeV
results_species = []

for label, primary, target in compositions:
    kernel._cross_section_cache.clear()
    model = models.AafragGammaSpectralModel(primary, target_composition=target)

    t0 = time.perf_counter()
    model(energy)
    t_cold = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(n_reps):
        model(energy)
    t_warm = (time.perf_counter() - t0) / n_reps

    n_pairs = len(target) * (len(primary) if isinstance(primary, dict) else 1)
    results_species.append((n_pairs, t_cold, t_warm))
    print(f"{label:24s}  cold={t_cold*1e3:8.1f} ms  warm={t_warm*1e3:7.3f} ms")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

grid_n, grid_cold, grid_warm = zip(*results_grid)
axes[0].plot(grid_n, np.array(grid_cold) * 1e3, "o-", label="cold (1st call)")
axes[0].plot(grid_n, np.array(grid_warm) * 1e3, "s-", label="warm (cache hit)")
axes[0].set_xlabel("output energy-grid points")
axes[0].set_ylabel("evaluate() cost [ms]")
axes[0].set_title("Cost vs. output grid size (1 species pair)")
axes[0].legend()
axes[0].grid(alpha=0.3)

pairs_n, pairs_cold, pairs_warm = zip(*results_species)
axes[1].plot(pairs_n, np.array(pairs_cold) * 1e3, "o-", label="cold (1st call)")
axes[1].plot(pairs_n, np.array(pairs_warm) * 1e3, "s-", label="warm (cache hit)")
axes[1].set_xlabel("number of (primary, target) species pairs")
axes[1].set_ylabel("evaluate() cost [ms]")
axes[1].set_title("Cost vs. composition size (30-point grid)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

- **Cold cost is flat vs. output grid size** (~0.95 s regardless of requesting 10 or 1000
  output points) but **scales linearly with the number of species pairs** (~0.95 s/pair,
  matching ADR-013's ~0.5-1 s/call `get_cross_section` estimate) -- confirming the cost
  driver is exactly what ADR-003 assumed: the cross-section computation, paid once per
  `(primary, target, channel, energy_grid)` key, not per requested output point.
- **Warm cost is ~2.6-2.7 ms per species pair**, also essentially flat vs. output grid size
  over the realistic 10-1000-point range gammapy energy axes actually use -- confirming
  `_interp_log_log`'s final interpolation is not a cost driver, and that `combine_species`'s
  cost is dominated by `get_spectrum`'s convolution (ADR-013's ~1 ms/call estimate), not
  Python-level overhead.
- **Concrete trigger point for revisiting ADR-002/ADR-003 (NumPy-only, no LUT/JAX):** a
  realistic `gammapy.modeling.Fit` run with, say, 4 species pairs costs ~9.4 ms/iteration
  once cross-sections are cached (confirmed identical order of magnitude in
  `05_gammapy_fit_recovery.ipynb`'s real `Fit.run()` call) -- a 10,000-iteration MCMC run
  would cost only ~94 s total. This assumption should be revisited once either (a) a
  realistic composition needs dozens of species pairs (warm cost would reach ~100 ms/iter,
  making large MCMC runs minutes-to-hours), or (b) the *cold* per-pair cost itself becomes
  the bottleneck for workflows that vary `target_composition` (not just fit `n_H`/primary
  parameters) across many likelihood evaluations, since ADR-008 assumed composition ratios
  are fixed, not fit.